In [120]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient


In [121]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

Activity.csv
Demographics.csv
Labels.csv
Physiology.csv
Sleep.csv


In [127]:
sleep_df1 = gen_date_col(data_dict['Sleep'],tcol='timestamp')
sleep_df1

,patient_id,timestamp,state,heart_rate,respiratory_rate,snoring,date
0,0f352,2019-06-25 22:53:00,AWAKE,69.0,14.0,False,2019-06-25
1,0f352,2019-06-25 22:54:00,AWAKE,66.0,14.0,False,2019-06-25
2,0f352,2019-06-25 22:55:00,AWAKE,70.0,14.0,False,2019-06-25
3,0f352,2019-06-25 22:56:00,AWAKE,70.0,13.0,False,2019-06-25
4,0f352,2019-06-25 22:57:00,AWAKE,68.0,13.0,False,2019-06-25
...,...,...,...,...,...,...,...
461418,f220c,2019-06-30 10:47:00,AWAKE,76.0,20.0,False,2019-06-30
461419,f220c,2019-06-30 10:48:00,AWAKE,73.0,21.0,False,2019-06-30
461420,f220c,2019-06-30 10:49:00,AWAKE,65.0,18.0,False,2019-06-30
461421,f220c,2019-06-30 10:50:00,AWAKE,75.0,15.0,False,2019-06-30


In [128]:
sleep_id_array = sleep_df1.patient_id.unique()
sleep_id_array

['0f352', '16f4b', '1fbe4', '30a32', '55cd4', ..., 'c8574', 'd7a46', 'e2472', 'ec812', 'f220c']
Length: 17
Categories (17, object): ['0f352', '16f4b', '1fbe4', '30a32', ..., 'd7a46', 'e2472', 'ec812', 'f220c']

# calculate sleep duration, number of bouts, durations for light, deep and REM

In [129]:
import pandas as pd
import datetime

summary_df_list =[]

for id_pick in sleep_id_array:
    print(id_pick)
    sleep_df_idPick = sleep_df1[sleep_df1['patient_id'] == id_pick] 
    
    date_array = sleep_df_idPick.date.unique()
    SLEEP_duration_list = []
    LIGHT_duration_list = []
    DEEP_duration_list = []
    REM_duration_list = []
    SLEEP_bouts_list = []
    date_list = []
    
    for target_date in date_array:
        date_list.append(target_date)
        # in each day
        filtered_df = sleep_df_idPick[sleep_df_idPick['date'] == target_date]
        
        # Assuming your dataframe is called sleep_df_idPick
        # First, make sure timestamp is a datetime object
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
    ##################### calculate sleep duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['state_change'] = (filtered_df['state'] != filtered_df['state'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['state_change', 'state']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        # Now sum duration per state, and convert to hours
        state_total_duration = chunk_durations.groupby('state')['duration_min'].sum() / 60  # in hours
        # print(state_total_duration)
        SLEEP_duration = state_total_duration['LIGHT'] + state_total_duration['DEEP'] + state_total_duration['REM']   
        
        # append data
        SLEEP_duration_list.append(SLEEP_duration)
        LIGHT_duration_list.append(state_total_duration['LIGHT']) 
        DEEP_duration_list.append(state_total_duration['DEEP'])
        REM_duration_list.append(state_total_duration['REM'])
        
    ##################### calculate sleep bouts ##########################  
        filtered_df_drop = filtered_df.copy()
        filtered_df_drop['state'] = filtered_df_drop['state'][filtered_df_drop['state'] != filtered_df_drop['state'].shift()]
        
        filtered_df_drop.dropna(subset=['state'], inplace=True)
        filtered_df_drop.reset_index(drop=True, inplace=True)
        
        
        def count_bouts(sleep_states):
            bouts = 0
            for i in range(len(sleep_states) - 2):
                # Check if there is a "light" -> "deep" -> "rem" sequence
                if sleep_states[i] == 'LIGHT' and sleep_states[i + 1] == 'DEEP' and sleep_states[i + 2] == 'REM':
                    bouts += 1
            return bouts
        
        
        # Apply the function to count bouts
        bouts = count_bouts(filtered_df_drop['state'])
        
        SLEEP_bouts_list.append(bouts)  
        
    sleep_summary = {
    'patient_id': id_pick,
    'date': date_list,
    'sleep_duration': SLEEP_duration_list,
    'light_duration': LIGHT_duration_list,
    'deep_duration': DEEP_duration_list,
    'REM_duration': REM_duration_list,
    'sleep_bout_count': SLEEP_bouts_list 
    }

    sleep_summary_df = pd.DataFrame(sleep_summary)  
    summary_df_list.append(sleep_summary_df)

0f352
16f4b
1fbe4
30a32
55cd4
76230
93c14
96adf
a2849
b0455
c55f8
c5785
c8574
d7a46
e2472
ec812
f220c


In [130]:
final_df = pd.concat(summary_df_list, ignore_index=True)

final_df.to_csv('output/processed_data/sleep_duration_bouts.csv', index=False)
final_df

,patient_id,date,sleep_duration,light_duration,deep_duration,REM_duration,sleep_bout_count
0,0f352,2019-06-25,0.316667,0.100000,0.216667,0.000000,0
1,0f352,2019-06-26,3.050000,1.800000,1.150000,0.100000,0
2,0f352,2019-06-27,3.950000,2.000000,1.683333,0.266667,0
3,0f352,2019-06-28,4.433333,3.200000,0.966667,0.266667,0
4,0f352,2019-06-29,6.000000,4.466667,0.533333,1.000000,0
...,...,...,...,...,...,...,...
830,f220c,2019-06-24,4.566667,3.816667,0.666667,0.083333,0
831,f220c,2019-06-25,4.033333,3.650000,0.000000,0.383333,0
832,f220c,2019-06-26,0.916667,0.666667,0.000000,0.250000,0
833,f220c,2019-06-29,0.300000,0.300000,0.000000,0.000000,0
